# 👑 KING2 + Gemma 4 E4B Fine-tuning
---
**تدريب Gemma 4 على شخصية KING2 ومعرفة الرياضيات**

يقوم هذا النوت بوك بـ:
1. تحميل معرفة KING2 من `knowledge_base.json`
2. تحويلها إلى بيانات تدريب
3. تدريب Gemma 4 باستخدام LoRA
4. حفظ النموذج النهائي `king2-gemma-4-e4b`

⚠️ يحتاج GPU (T4 في Colab مجاني يكفي)

In [ ]:
# @title 1. تثبيت الاعتماديات
# Install required packages
!pip install -qU transformers datasets accelerate peft bitsandbytes
!pip install -qU trl
!pip install -qU --upgrade huggingface_hub

print('Dependencies installed!')

In [ ]:
# @title 2. تثبيت Unsloth (أسرع تدريب)
import torch
major_version, minor_version = torch.__version__.split('.')[:2]
print(f'PyTorch version: {torch.__version__}')

try:
    import unsloth
    print('Unsloth already installed!')
except:
    print('Installing Unsloth...')
    !pip install -qU unsloth
    !pip install -qU --force-reinstall --no-deps unsloth
    print('Unsloth installed!')

In [ ]:
# @title 3. تحميل بيانات KING2
import json
import os

# نقوم بتحميل knowledge_base.json من المشروع
# ارفع الملف أولاً أو استخدم الرابط
from google.colab import files
print('⚠️ الرجاء رفع knowledge_base.json من مشروع KING2')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Loaded: {filename}')
    with open(filename, 'r', encoding='utf-8') as f:
        kb = json.load(f)
    break

# Extract math entries
math_cats = {'حساب', 'جبر', 'هندسة', 'إحصاء', 'تفاضل وتكامل', 'رياضيات'}
math_entries = [e for e in kb.get('knowledge', []) if e.get('category') in math_cats]
general_entries = [e for e in kb.get('knowledge', []) if e.get('category') not in math_cats]

print(f'Math entries: {len(math_entries)}')
print(f'General entries: {len(general_entries)}')
print(f'Total: {len(kb["knowledge"])}')

In [ ]:
# @title 4. إعداد بيانات التدريب (KING2 Format)
import pandas as pd

# شخصية KING2 - هوية النظام
system_prompt = (
    'أنت KING2، مساعد الذكاء الاصطناعي الملكي المتطور. '
    'أجب بالعربية الفصحى أولاً دائماً. '
    'كن مختصراً ومفيداً وواضحاً. '
    'استخدم markdown عند الحاجة. '
    'للمسائل الرياضية: اشرح الخطوات بالتفصيل. '
    'إذا سألك المستخدم بلغة أخرى، أجب بنفس اللغة.'
)

def create_alpaca_format(question, answer):
    return {
        'instruction': system_prompt,
        'input': question,
        'output': answer
    }

# إنشاء بيانات التدريب
training_data = []

# 1. المدخلات الرياضية
for entry in math_entries:
    training_data.append(create_alpaca_format(
        entry['question'],
        entry['answer']
    ))

# 2. مدخلات عامة (شخصية KING2)
general_qa = [
    {'q': 'من أنت؟', 'a': 'أنا KING2، مساعد الذكاء الاصطناعي الملكي المتطور. أنا هنا لمساعدتك في أي شيء تحتاجه.'},
    {'q': 'ما اسمك؟', 'a': 'اسمي KING2، ويعني الملك الذهبي. أنا مساعد ذكي تم تطويره لخدمتك.'},
    {'q': 'بأي لغة تتحدث؟', 'a': 'أتحدث العربية أولاً وبشكل أساسي، ولكن يمكنني التحدث بالإنجليزية والصينية والعديد من اللغات الأخرى.'},
    {'q': 'ماذا تستطيع أن تفعل؟', 'a': 'أستطيع مساعدتك في: الحساب والرياضيات، البرمجة، التحليل، الترجمة، الكتابة الإبداعية، البحث في الإنترنت، وتحليل الصور.'},
    {'q': 'من صنعك؟', 'a': 'أنا KING2، تم تطويري من قبل فريق KING2 AI ليكون أول مساعد ذكاء اصطناعي عربي متكامل.'},
    {'q': 'ما هي مجالات خبرتك؟', 'a': 'خبرتي تشمل الرياضيات (الجبر، الهندسة، التفاضل والتكامل، الإحصاء)، البرمجة (Python، JavaScript)، الترجمة، التحليل، والبحث.'},
    {'q': 'كيف تقوم بحل المسائل الرياضية؟', 'a': 'أقوم بحل المسائل الرياضية خطوة بخطوة، أشرح كل خطوة بالتفصيل مع الأمثلة لضمان الفهم الكامل.'},
    {'q': 'هل يمكنك مساعدتي في الواجبات المدرسية؟', 'a': 'نعم، يمكنني مساعدتك في فهم المفاهيم وحل التمارين. اشرح لي السؤال وسأبسطه لك خطوة بخطوة.'},
]

for item in general_qa:
    training_data.append(create_alpaca_format(item['q'], item['a']))

print(f'Training samples: {len(training_data)}')

In [ ]:
# @title 5. تجهيز البيانات للـ Trainer
from datasets import Dataset

def format_chat_template(example):
    """Convert to Gemma 4 chat format."""
    prompt = f"<bos><start_of_turn>user\n{example['instruction']}\n\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    return {'text': prompt}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_chat_template)

print(f'Dataset size: {len(dataset)}')
print(f'Example:\n{dataset[0]["text"][:300]}...')

In [ ]:
# @title 6. تحميل Gemma 4 E4B
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = 'google/gemma-4-e4b'  # Gemma 4

# 4-bit quantization لتوفير الذاكرة
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f'Loading {model_name} in 4-bit...')

try:
    # Try with Unsloth first (faster)
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        load_in_4bit=True,
        max_seq_length=2048,
    )
    print('Loaded with Unsloth!')
    unsloth_available = True
except Exception as e:
    print(f'Unsloth failed: {e}')
    print('Loading with standard Transformers...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = 'right'
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16
    )
    unsloth_available = False

print(f'Model loaded! Parameters: {model.num_parameters() / 1e9:.2f}B')

In [ ]:
# @title 7. إعداد LoRA
from peft import LoraConfig, get_peft_model, TaskType

if unsloth_available:
    from unsloth import is_bfloat16_supported
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.1,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
        use_rslora=True
    )
else:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.1,
        bias='none',
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)

print('LoRA configured!')
print(f'Trainable parameters: {model.num_parameters(only_trainable=True) / 1e6:.2f}M')

In [ ]:
# @title 8. التدريب
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./king2-gemma-4-e4b-lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    logging_steps=5,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=2048,
    dataset_text_field='text',
    packing=False,
)

print('Starting training...')
print(f'Batch size: {training_args.per_device_train_batch_size}')
print(f'Gradient accumulation: {training_args.gradient_accumulation_steps}')
print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'Epochs: {training_args.num_train_epochs}')

trainer.train()

print('Training complete!')

In [ ]:
# @title 9. حفظ النموذج
import shutil
import os

# حفظ LoRA adapter
model.save_pretrained('./king2-gemma-4-e4b-lora/final')
tokenizer.save_pretrained('./king2-gemma-4-e4b-lora/final')

# إنشاء مجلد مضغوط للتحميل
!zip -r king2-gemma-lora.zip ./king2-gemma-4-e4b-lora/

print('\n✅ Model saved!')
print('📁 LoRA adapter: ./king2-gemma-4-e4b-lora/final')
print('📦 Zip: king2-gemma-lora.zip')
print('\n⬇️ تحميل الملف:')
from google.colab import files
files.download('king2-gemma-lora.zip')

In [ ]:
# @title 10. تجربة النموذج
def ask_king2(prompt):
    if unsloth_available:
        FastLanguageModel.for_inference(model)
    
    full_prompt = (
        f'<bos><start_of_turn>user\n'
        f'{system_prompt}\n\n{prompt}<end_of_turn>\n'
        f'<start_of_turn>model\n'
    )
    
    inputs = tokenizer(full_prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split('model\n')[-1] if 'model\n' in response else response
    return response

# اختبارات
tests = [
    'من أنت؟',
    'ما هو الجمع في الرياضيات؟',
    'اشرح لي نظرية فيثاغورس',
]

for test in tests:
    print(f'\n❓ {test}')
    print(f'👑 {ask_king2(test)[:200]}...')
    print('---')

In [ ]:
# @title 11. (اختياري) دمج LoRA مع النموذج الأساسي
# هذا الخيار يدمج LoRA مع النموذج الأصلي لتوليد GGUF يمكن استخدامه مع LM Studio

merge = input('هل تريد دمج LoRA مع النموذج الأصلي؟ (y/n): ')

if merge.lower() == 'y':
    print('Merging LoRA with base model...')
    
    # Merge
    merged_model = model.merge_and_unload()
    
    # Save merged
    merged_model.save_pretrained('./king2-gemma-4-e4b-merged', safe_serialization=True)
    tokenizer.save_pretrained('./king2-gemma-4-e4b-merged')
    
    # Zip for download
    !zip -r king2-gemma-merged.zip ./king2-gemma-4-e4b-merged/
    
    print('\n✅ Merged model saved!')
    print('📁 Path: ./king2-gemma-4-e4b-merged/')
    
    # Download
    from google.colab import files
    files.download('king2-gemma-merged.zip')
else:
    print('Skipping merge. You can use the LoRA adapter separately.')

---
## 📝 ملخص

**الخطوات التالية بعد التدريب:**
1. انسخ الملف `king2-gemma-lora.zip` أو `king2-gemma-merged.zip` إلى مشروع KING2
2. إذا كان LoRA: load adapter عند تشغيل Gemma 4
3. إذا كان Merged: استخدمه مباشرة مع LM Studio أو llama.cpp
4. اختبر النموذج مع أسئلة الرياضيات

**الملفات المنتجة:**
- LoRA adapter: `./king2-gemma-4-e4b-lora/`
- Merged model: `./king2-gemma-4-e4b-merged/` (اختياري)

**لمساعدة إضافية:** راسلني في المحادثة!